# ECE 539 — P4 Mood Music Recommender Integration
**Team G16 | P4: Seungjae Chung**

This notebook implements the **P4 integration layer** from the team proposal: combine predicted valence/arousal (P2) and prompt-specific CLAP audio–text scores (P3), rank songs, and display the top 10 for a selected mood/activity.

**Run immediately:** `DATA_MODE = "demo"` uses **synthetic data** to test the interface and ranking logic; these are **not actual DEAM results**. No GPU or Drive needed.

**Integrate later:** `DATA_MODE = "upload"` accepts model-output CSVs; see the expected schema below. Use the team's `song_split_ids.csv` to preserve song-level splits and avoid evaluating on test data until final evaluation.

**Source alignment:** Proposal: DEAM 500-song subset, MERT emotion predictor, LAION-CLAP activity scorer, top-10 ranker. P1 baseline can replace P2 temporarily if exported predictions are available. No P2/P3 performance claims are made here.

## 1. Imports and settings
Run all cells in order. Colab already includes pandas, numpy, and ipywidgets in most runtimes; the notebook falls back to a text selector if widgets are unavailable.

In [1]:
import io
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

SEED = 539
DATA_MODE = "demo"  # "demo" or "upload"
TOP_K = 10
EMOTION_WEIGHT = 0.5  # Initial prototype weight; tune using DEV only
EVALUATION_SPLIT = "dev"  # Do not use held-out test split for tuning

# Provisional prototype targets on the 1–9 DEAM-style emotion scale.
# Confirm exact P2 label scaling and target mapping before final evaluation.
MOODS = {
    "Relax":   {"valence": 6.5, "arousal": 2.5, "prompt": "calm, relaxing music"},
    "Study":   {"valence": 5.5, "arousal": 3.0, "prompt": "gentle, unobtrusive music for studying"},
    "Workout": {"valence": 7.0, "arousal": 8.0, "prompt": "energetic music for exercising"},
    "Happy":   {"valence": 8.0, "arousal": 6.5, "prompt": "happy, upbeat music"},
    "Sad":     {"valence": 2.5, "arousal": 3.0, "prompt": "sad, reflective music"},
}
print("Mode:", DATA_MODE, "| Mood choices:", list(MOODS))

Mode: demo | Mood choices: ['Relax', 'Study', 'Workout', 'Happy', 'Sad']


## 2. Data input
### CSV schema for real integration

Upload **one combined CSV** with these columns:

| Column | Meaning |
|---|---|
| `song_id` | DEAM song ID (must be unique) |
| `valence` | P2 predicted valence (or P1 Ridge prediction temporarily) |
| `arousal` | P2 predicted arousal (or P1 Ridge prediction temporarily) |
| `clap_Relax`, `clap_Study`, `clap_Workout`, `clap_Happy`, `clap_Sad` | P3 audio–text scores for each corresponding prompt |
| `split` (optional) | `train`, `dev`, or `test`; dev used for development |

**Important:** CLAP scores must be **different per prompt**. A single `clap_score` shared across moods does not represent mood-specific audio–text matching. If teammates supply separate files, join on `song_id` and validate IDs before uploading. Do not replace missing CLAP scores with random values in real-data mode.

The demo generates 100 synthetic song IDs and synthetic predictions; no real music or ground-truth labels are used.

In [2]:
def make_demo_data(n=100, seed=SEED):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        "song_id": np.arange(1, n + 1),
        "valence": rng.uniform(1, 9, n),
        "arousal": rng.uniform(1, 9, n),
        "split": ["dev"] * n,
    })
    # Mock prompt-specific CLAP scores with some dependence on target emotion,
    # plus noise; ONLY for testing ranker behavior, NOT model evaluation.
    for mood, cfg in MOODS.items():
        dist = np.sqrt(((df["valence"] - cfg["valence"]) / 8)**2 +
                       ((df["arousal"] - cfg["arousal"]) / 8)**2) / np.sqrt(2)
        df[f"clap_{mood}"] = np.clip(0.85 - dist + rng.normal(0, 0.12, n), 0, 1)
    return df

def upload_combined_csv():
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Upload mode is intended for Google Colab. Use demo mode locally.") from exc
    print("Choose a combined CSV with song_id, valence, arousal and clap_<Mood> columns.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one CSV file.")
    filename, content = next(iter(uploaded.items()))
    if not filename.lower().endswith(".csv"):
        raise ValueError("File must be a CSV.")
    return pd.read_csv(io.BytesIO(content))

songs = make_demo_data() if DATA_MODE == "demo" else upload_combined_csv()
print("Loaded", len(songs), "rows; mode:", DATA_MODE)
display(songs.head())

Loaded 100 rows; mode: demo


,song_id,valence,arousal,split,clap_Relax,clap_Study,clap_Workout,clap_Happy,clap_Sad
0,1,1.146640,3.139947,dev,0.255876,0.458771,0.234927,0.318002,0.817662
1,2,4.486465,4.711755,dev,0.487853,0.658483,0.263783,0.525660,0.616104
2,3,6.626764,8.807995,dev,0.256142,0.456642,0.741340,0.715791,0.091176
3,4,3.216836,2.991122,dev,0.627609,0.568440,0.228875,0.259379,0.848456
4,5,8.920674,8.868106,dev,0.354019,0.221323,0.651691,0.783106,0.110911


## 3. Validate data and calculate scores

**Emotion match:** normalized Euclidean distance in 2D valence/arousal space (1–9). Similarity = `1 − distance`.

**CLAP match:** per-mood raw CLAP scores are min–max normalized across the candidate pool. This is a prototype ranking choice, **not** calibrated probability. Constant scores get a neutral 0.5.

**Combined score:** `emotion_weight × emotion_similarity + (1 − emotion_weight) × clap_similarity`. Rank descending, break ties by `song_id`. Restrict to the development split if split labels are supplied. We do not claim this weighting is optimal.

In [3]:
def validate_songs(df):
    required = ["song_id", "valence", "arousal"] + [f"clap_{m}" for m in MOODS]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    if df["song_id"].isna().any() or df["song_id"].duplicated().any():
        raise ValueError("song_id must be nonmissing and unique; check joins and split files")
    for col in required[1:]:
        values = pd.to_numeric(df[col], errors="coerce")
        if not np.isfinite(values.to_numpy(dtype=float)).all():
            raise ValueError(f"{col} contains missing or nonnumeric values")
    if ((df[["valence", "arousal"]].astype(float) < 1) |
        (df[["valence", "arousal"]].astype(float) > 9)).any().any():
        raise ValueError("Valence/arousal must be on the 1–9 scale. Convert model outputs first.")
    if "split" in df.columns:
        labels = df["split"].astype(str).str.lower()
        if not labels.isin(["train", "dev", "test"]).all():
            raise ValueError("split must contain only train, dev or test")
    return True

def minmax_normalize(values):
    x = np.asarray(values, dtype=float)
    lo, hi = x.min(), x.max()
    return np.full_like(x, 0.5) if np.isclose(lo, hi) else (x - lo) / (hi - lo)

def recommend(df, mood="Relax", k=TOP_K, emotion_weight=EMOTION_WEIGHT, split=EVALUATION_SPLIT):
    validate_songs(df)
    if mood not in MOODS:
        raise ValueError(f"Unknown mood: {mood}")
    if not 0 <= emotion_weight <= 1:
        raise ValueError("emotion_weight must be between 0 and 1")
    if k < 1:
        raise ValueError("k must be positive")
    candidates = df.copy()
    if "split" in candidates.columns:
        candidates = candidates[candidates["split"].astype(str).str.lower() == split].copy()
    if len(candidates) < k:
        raise ValueError(f"Need at least {k} songs in split '{split}', found {len(candidates)}")
    cfg = MOODS[mood]
    distance = np.sqrt(((candidates["valence"].astype(float) - cfg["valence"]) / 8)**2 +
                       ((candidates["arousal"].astype(float) - cfg["arousal"]) / 8)**2) / np.sqrt(2)
    candidates["emotion_similarity"] = 1 - distance
    candidates["clap_similarity"] = minmax_normalize(candidates[f"clap_{mood}"].astype(float))
    candidates["combined_score"] = (emotion_weight * candidates["emotion_similarity"] +
                                    (1 - emotion_weight) * candidates["clap_similarity"])
    candidates["mood"] = mood
    candidates = candidates.sort_values(["combined_score", "song_id"], ascending=[False, True])
    cols = ["song_id", "mood", "valence", "arousal", f"clap_{mood}",
            "emotion_similarity", "clap_similarity", "combined_score"]
    return candidates[cols].head(k).reset_index(drop=True)

validate_songs(songs)
print("Validation passed.")
display(recommend(songs, "Relax"))

Validation passed.


,song_id,mood,valence,arousal,clap_Relax,emotion_similarity,clap_similarity,combined_score
0,75,Relax,7.246169,2.346360,0.903766,0.932664,0.989243,0.960953
1,73,Relax,6.701050,1.040945,0.913462,0.869818,1.000000,0.934909
2,45,Relax,7.825376,4.238135,0.861144,0.806800,0.941962,0.874381
3,77,Relax,6.456299,2.175838,0.704682,0.971089,0.768393,0.869741
4,80,Relax,5.447780,3.196317,0.773666,0.888476,0.844920,0.866698
5,87,Relax,7.493004,1.299998,0.793249,0.862328,0.866643,0.864486
6,85,Relax,7.587145,3.813780,0.799927,0.849275,0.874052,0.861663
7,28,Relax,6.102981,1.385452,0.752081,0.895423,0.820974,0.858199
8,27,Relax,8.360718,1.604319,0.819237,0.817472,0.895473,0.856472
9,51,Relax,4.433708,2.357877,0.812850,0.816932,0.888387,0.852660


## 4. Interactive mood selector
Choose a mood; the output displays the 10 highest-scoring unique song IDs. These are **synthetic IDs in demo mode**, not playable track recommendations.

In [4]:
try:
    import ipywidgets as widgets
    from IPython.display import clear_output
    dropdown = widgets.Dropdown(options=list(MOODS), value="Relax", description="Mood:")
    slider = widgets.FloatSlider(value=EMOTION_WEIGHT, min=0, max=1, step=0.1,
                                 description="Emotion weight:", style={"description_width": "initial"})
    out = widgets.Output()
    def show_results(change=None):
        with out:
            clear_output(wait=True)
            print("Mode:", DATA_MODE, "| Prompt:", MOODS[dropdown.value]["prompt"])
            display(recommend(songs, dropdown.value, emotion_weight=slider.value))
    dropdown.observe(show_results, names="value")
    slider.observe(show_results, names="value")
    display(widgets.VBox([dropdown, slider, out]))
    show_results()
except ImportError:
    print("ipywidgets unavailable; choose a mood manually:")
    display(recommend(songs, "Study"))

## 5. Automated integration tests
Tests verify valid rankings, distinct IDs, descending scores, mood sensitivity, and input errors. Passing tests demonstrate **software functionality only**, not recommendation accuracy.

In [5]:
for mood in MOODS:
    result = recommend(songs, mood)
    assert len(result) == TOP_K
    assert result["song_id"].is_unique
    assert result["combined_score"].is_monotonic_decreasing
    assert result["combined_score"].between(0, 1).all()

assert recommend(songs, "Relax")["song_id"].tolist() != recommend(songs, "Workout")["song_id"].tolist(), "Demo should respond to mood"
try:
    recommend(songs, "InvalidMood")
    raise AssertionError("Invalid mood was not rejected")
except ValueError:
    pass
try:
    validate_songs(songs.drop(columns=["clap_Relax"]))
    raise AssertionError("Missing CLAP column was not rejected")
except ValueError:
    pass
try:
    validate_songs(pd.concat([songs, songs.iloc[[0]]], ignore_index=True))
    raise AssertionError("Duplicate song ID was not rejected")
except ValueError:
    pass
print(f"PASS: all {len(MOODS)} moods return {TOP_K} unique ranked songs; error checks pass.")

PASS: all 5 moods return 10 unique ranked songs; error checks pass.


## 6. Export results
Export top-10 lists for each mood. **Demo exports are synthetic and must not be reported as real model results.**

In [6]:
all_results = pd.concat([recommend(songs, mood) for mood in MOODS], ignore_index=True)
filename = "P4_DEMO_top10.csv" if DATA_MODE == "demo" else "P4_DEV_top10.csv"
all_results.to_csv(filename, index=False)
print(f"Saved {filename} with {len(all_results)} rows")
display(all_results.head())
# Optional in Colab: uncomment to download
# from google.colab import files
# files.download(filename)

Saved P4_DEMO_top10.csv with 50 rows


,song_id,mood,valence,arousal,clap_Relax,emotion_similarity,clap_similarity,combined_score,clap_Study,clap_Workout,clap_Happy,clap_Sad
0,75,Relax,7.246169,2.346360,0.903766,0.932664,0.989243,0.960953,NaN,NaN,NaN,NaN
1,73,Relax,6.701050,1.040945,0.913462,0.869818,1.000000,0.934909,NaN,NaN,NaN,NaN
2,45,Relax,7.825376,4.238135,0.861144,0.806800,0.941962,0.874381,NaN,NaN,NaN,NaN
3,77,Relax,6.456299,2.175838,0.704682,0.971089,0.768393,0.869741,NaN,NaN,NaN,NaN
4,80,Relax,5.447780,3.196317,0.773666,0.888476,0.844920,0.866698,NaN,NaN,NaN,NaN


## 7. Integration checklist and report notes

- [x] Implemented score normalization and combined emotion/CLAP ranker
- [x] Implemented mood dropdown and top-10 display
- [x] Added validation, automated tests, and CSV export
- [ ] Obtain P1 `song_split_ids.csv` and match song IDs/splits
- [ ] Obtain P2 MERT/MLP **predicted** valence/arousal per song
- [ ] Obtain P3 **per-prompt** CLAP similarity scores per song
- [ ] Join real outputs on `song_id`, check no duplicates/missing IDs, and rerun in `upload` mode
- [ ] Tune mood targets, weights, and CLAP prompts using train/dev only
- [ ] Evaluate mood-hit@10 and test-split results **once after model selection**

**Accurate progress-report wording after running the demo:** “P4 implemented a modular mood-based ranking prototype with dropdown selection, top-10 output, validation tests, and CSV export. Software behavior was tested using synthetic inputs; integration with actual P2/P3 predictions remains pending.”

**Note:** This notebook does not run MERT or CLAP, does not infer emotion from raw audio, and does not provide actual recommendation-quality metrics. The team's proposal defines final test-set metrics and evaluation separately.